# Actividad 4 - Comparación de tres modelos de Hugging Face
**Caso:** comentarios de clientes de Supermercados Miroslava (empresa ficticia).  
**Rama:** procesamiento de lenguaje natural (PLN), análisis de sentimiento.  
**Clases:** POS (positivo), NEG (negativo), NEU (neutral).

El cuaderno descarga **tres modelos preentrenados**, los evalúa con el mismo dataset y calcula *accuracy*, *precision*, *recall*, *F1* y latencia. Los datos son **simulados**, no reales. Se necesitan Internet y preferentemente GPU. No hay métricas inventadas ni salidas prellenadas.

## Paso 1. Preparar Google Colab
En Colab elige **Entorno de ejecución > Cambiar tipo de entorno de ejecución > T4 GPU** (si está disponible). Ejecuta esta celda para instalar dependencias.

In [ ]:
%pip -q install "transformers>=4.45,<5.0" "datasets>=3.0,<5.0" "huggingface_hub>=0.34,<1.0" "accelerate>=0.28,<2.0" "scikit-learn>=1.4,<2.0" "pandas>=2,<3" "matplotlib>=3.8,<4" "sentencepiece>=0.2" "safetensors>=0.4" "python-docx>=1.1"

## Paso 2. Verificar GPU y versiones
La inferencia también puede correr en CPU, pero tardará más y no cumpliría la condición de GPU de la actividad.

In [ ]:
import os, sys, time, gc, json, hashlib, platform, subprocess
from pathlib import Path
from datetime import datetime, timezone
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import transformers, datasets, huggingface_hub
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline, set_seed
from datasets import load_dataset
from huggingface_hub import HfApi, login

set_seed(42)
GPU = torch.cuda.is_available()
DEVICE = 0 if GPU else -1
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("GPU disponible:", GPU)
print("Dispositivo:", torch.cuda.get_device_name(0) if GPU else "CPU (cambiar entorno en Colab)")
if GPU:
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("AVISO: se recomienda activar T4 GPU antes de continuar.")
Path("results").mkdir(exist_ok=True)
Path("data").mkdir(exist_ok=True)

## Paso 3. Autenticación en Hugging Face Hub
Los modelos son públicos. Para dejar evidencia de autenticación, crea un token de **lectura** en https://huggingface.co/settings/tokens y guárdalo en **Secretos de Colab** con el nombre `HF_TOKEN`. No escribas el token en el notebook ni lo subas a GitHub. Si no hay token, se usan las descargas públicas sin iniciar sesión.

In [ ]:
TOKEN = os.environ.get("HF_TOKEN")
if not TOKEN:
    try:
        from google.colab import userdata
        TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        TOKEN = None
if TOKEN:
    login(token=TOKEN, add_to_git_credential=False)
    print("Autenticacion de Hugging Face realizada. El token no se muestra.")
else:
    print("Sin token: se accede a modelos publicos de Hugging Face.")
    print("Para autenticarte manualmente puedes ejecutar notebook_login().")
    # from huggingface_hub import notebook_login
    # notebook_login()

## Paso 4. Cargar el dataset
Este notebook lleva una copia integrada del CSV para que funcione en Colab sin subir archivos adicionales. El archivo también está disponible en `data/comentarios_clientes.csv` y se regenera con `src/generar_dataset.py`. **Origen: 60 comentarios inventados para una actividad académica** (20 por clase). Se carga usando la biblioteca **Hugging Face Datasets**.

In [ ]:
import base64, zlib
DATOS_COMPRIMIDOS_B64 = "eNqFWMmO20YQvQfIP/QHEIbWWU4+OJPk4EycxR/QJFuaHjS76V4Ej/9mjj7MIfDNlwDhj+VVNSlRjsgANjDWSKWqt1XRui6i+hhdoaL+kFSU+GcjC+f1Xtnvv3uzWCyLX5SQUdlaK++saNKT8N1zq2snnkTFr1QPUshGltroWtavine//lHQRyrtbBF0k4ysHVdbFXdGtKqmTxuj9t0XlGhaoyJVkzaqIGoljBQPzkvRenXQIcpcUtno1V6eV1wXb2XAG13j6Ovy5x/ck2jQ9lOqJTWN9lDPS58Ljd59XmxT3FmMFH33WURXuyCM4+J1qiJ+AkTCqkoFHWXZd6XtAY1Jr915rS0aE7I1upKV7r5YgdYCYWIw3ZNoE/ps5V56EbSl7ygNoA89em17Xu2KqlXyUQGVHbogGghyo0SdPAFHDUtCjgD1co6F6+Lt2Vg7r0JFw4IRxkvhz8dKGUV1FQio3QwFNySRfQoRbBJCqNZCSuhKKAM6QpXwOQd6E5o51ppk4bb4WZbdixQh7XSluYlG+Urail59Ymq4rVIK52tljzUnyFguSHeQaSxdpP6cOYAT0WigB9Qwb+sIgaD2yYL3GfSWy+J3VWk0SB/PYENgUaumJRGXmtBrWlnJeka4y1VPAmoRBSm7Sz0633vAMZiEpT+KrX/7eaV18SPeuOueoS2MwgKGrCAFo6oHhZ5AQyuZVsOd2npevMsNya3tnvfakuw8GfXk+qh8oy08cgTgsmqX22x3H5yVhhypPpIh2PWAXBK5UITAK40TO1nFdDTpZfCvSGv4BRSPKqR2yaGTU4OknKIMvQyZpRkKrlkVqSV37hJ/G4o+CVfGdFAiWVFCuScFz+t2eUOgZXQHdRrdtJoki07zQPm71Mlfc9PeUoOkSd1otlCtjvFJRSoDu6I6pUH3NWpzmYcV5B8iMjHIqMMOonBMgKLi/gCPUZEDTKEo+YYImSNihb2gv0lybUdZbh3x3S8N+C3x6K+K+7ufLpKxWn2blwHzKe9R+BHZgo6ThDWFLslunJy52n+mXbN4M0tUaMgdSU2FIZ9QF9wiUmUuM0XrasMi7sNyIBbmTvjLbiCtkOBqlSE0LlecMNcKriBTA+maEp8SEfMgRVix2ue91XOdvdNPepGJqyEu8yLead4vgNxBwbw0jnLEilY+yrlq7Amv0E8EtX7glhSGUWpKE9pXEnt5IOAinbwUKld6Xii0Sg+0OElzjQ5wO76ge6nSANalZFtl/eMCAE7Ag+O/1qF1VtPmAzqOOH3I8w8czaK/XpA6mKp0ciMLV1NiizznsAb58kHcqRnM1svxdkE/ZGvcS3nBYDDslEiFSXiu/8Usp+tshmNgtCAiKtN93jnrjvsfNeaoXK/PNgzJSVoJoRrqsXKwq+792YcB0TVDx3qTw/egGZ1kR674kPTBsSEAWjMkw7Q+1tvinhMiCwnlyEGZSHJlU7It0BQdgiBxblA+jvpvyQbHAgxAmXDLKx2HQgWrzUbQ+nq89foAU4+os8MLRn/KS3XYeJeyZ31TvOftb5TsM3rvXSCd9vEFYHA5aLtzvpGjTLw82e1oQ1GKybM9Bf1TsfGRNZ9lmwVfy8goOhpHB18tu78g/pCDm2/TBFoM3aNTcG2WxW+J9IO1UiqkFihEpvHhLkuvxrfG/d37yxNuVsU/f/+A8egaTqrmxRSSiYz1+WF/oIcSPCG85nKTM65Hmwm51z1Tkb3HtTzkbEPT2r6ti6Ntivt85jsKUizI3M9wotC1Mbp9pqfbYro/gYGyp02LT7sGPxk9mOUTIv71dDNXJILeBgCp5KCi2agNm9RB9c9M/VyCxP7g5vq6JuqCpqeJTF7QzBUlnvfu0yhlc5mJNN3c8HKkVMGRLoEuYCNceLSW98hJ5RPz3QKkN6l7NjQSxmid6V4ihqSxhqQmnl9PT7RdnB9fYL2iR4YeGYnucSaiI0q9NofvVKkc5y7wvUV5172442yKkxJJnhrVD3UpK7ek63dZ0BxmWT6KNjQXZRymSd+uM7D7/mLZYwn6/m4GDY94vmJ0us+ljm6mj82J6XHoQNaDRU/VWuwpF17N2mu7HS87JGTjcmfkVteensH5iJmD+eqbMy0/VUiOMxjEYnTKb0NRjMeY/2nr+vhUlugxygY8Kw97zXZfG8opGvG4lyZwvwFvdzZHWYuNYUyfIpb/LwCiUtIMUpxwxba/23vD12q0yrM2R0dMfijrt84MXleLUyZRCp0WUq1BfK5+ktbUiP8C/5pyOw=="
DATOS_SHA256 = "b89f4629f9d0f19d173273061b0c1a9b14835d7994971c845f03984b57ad9cfd"
archivo_csv = Path("data/comentarios_clientes.csv")
archivo_csv.write_bytes(zlib.decompress(base64.b64decode(DATOS_COMPRIMIDOS_B64)))
assert hashlib.sha256(archivo_csv.read_bytes()).hexdigest() == DATOS_SHA256
hf_dataset = load_dataset("csv", data_files={"test": str(archivo_csv)})["test"]
df = hf_dataset.to_pandas()
assert len(df) == 60
assert df["etiqueta"].value_counts().to_dict() == {"POS": 20, "NEG": 20, "NEU": 20}
print("Datos cargados:", len(df), "comentarios")
print(df["etiqueta"].value_counts().to_string())
display(df.head(6))

## Ecosistema Hugging Face
- **Models:** repositorio de pesos y configuraciones de modelos (https://huggingface.co/models).
- **Datasets:** biblioteca y catálogo de conjuntos de datos (https://huggingface.co/datasets).
- **Spaces:** demostraciones de aplicaciones de IA (https://huggingface.co/spaces). No se requiere publicar una para esta tarea.

## Paso 5. Tres modelos y criterios de comparación
Todos hacen clasificación de sentimiento con tres etiquetas. Se usa el mismo conjunto de 60 comentarios, sin entrenamiento ni ajuste de pesos. El costo se aproxima mediante tamaño y uso de GPU; no se inventan precios comerciales.

**Licencia:** DistilBERT declara Apache-2.0. En los otros dos modelos no aparece una licencia inequívoca en la ficha revisada; deben validarse los permisos antes de usarlos comercialmente.

In [ ]:
MODELOS = [
    {"id": "pysentimiento/robertuito-sentiment-analysis", "nombre": "RoBERTuito", "licencia_revisada": "No declarada en ficha", "licencia_clara": False},
    {"id": "lxyuan/distilbert-base-multilingual-cased-sentiments-student", "nombre": "DistilBERT multilingue", "licencia_revisada": "Apache-2.0", "licencia_clara": True},
    {"id": "cardiffnlp/twitter-xlm-roberta-base-sentiment", "nombre": "XLM-RoBERTa", "licencia_revisada": "No declarada en ficha", "licencia_clara": False},
]
display(pd.DataFrame(MODELOS)[["nombre", "id", "licencia_revisada"]])

## Paso 6. Funciones para las métricas
Se reporta **precision, recall y F1 macro** para dar el mismo peso a las tres clases. Además, se mide *recall* de NEG porque detectar quejas es importante para el caso de uso.

In [ ]:
"""Funciones sencillas para comparar modelos de sentimiento."""
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


def normalizar_etiqueta(etiqueta):
    """Convierte etiquetas publicadas por los modelos a POS/NEG/NEU."""
    texto = str(etiqueta).strip().upper()
    equivalencias = {
        "POS": "POS", "POSITIVE": "POS", "POSITIVO": "POS",
        "NEG": "NEG", "NEGATIVE": "NEG", "NEGATIVO": "NEG",
        "NEU": "NEU", "NEUTRAL": "NEU", "NEUTRO": "NEU",
    }
    if texto not in equivalencias:
        raise ValueError(f"Etiqueta desconocida: {etiqueta}. Revisar config.json del modelo.")
    return equivalencias[texto]


def calcular_metricas(reales, predichas):
    etiquetas = ["NEG", "NEU", "POS"]
    return {
        "accuracy": float(accuracy_score(reales, predichas)),
        "precision_macro": float(precision_score(reales, predichas, labels=etiquetas, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(reales, predichas, labels=etiquetas, average="macro", zero_division=0)),
        "f1_macro": float(f1_score(reales, predichas, labels=etiquetas, average="macro", zero_division=0)),
        "recall_neg": float(recall_score(reales, predichas, labels=["NEG"], average="macro", zero_division=0)),
    }

## Paso 7. Inferencia, latencia y recursos
Se descarga y evalúa **un modelo a la vez** para evitar saturar la memoria de la GPU. La latencia es el tiempo promedio por comentario con peticiones individuales, **sin incluir descarga ni carga del modelo**. Los primeros ejemplos se usan como calentamiento. Los resultados reales aparecen al ejecutar esta celda.

In [ ]:
textos = df["texto"].astype(str).tolist()
verdaderas = df["etiqueta"].astype(str).tolist()
filas_metricas = []
filas_predicciones = []
revisiones = {}
errores = {}
api = HfApi()

for spec in MODELOS:
    modelo_id = spec["id"]
    print("\nEvaluando:", spec["nombre"], "-", modelo_id)
    try:
        if GPU:
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
        t_carga = time.perf_counter()
        tokenizer = AutoTokenizer.from_pretrained(modelo_id, trust_remote_code=False)
        model = AutoModelForSequenceClassification.from_pretrained(modelo_id, trust_remote_code=False)
        clasificador = pipeline("text-classification", model=model, tokenizer=tokenizer, device=DEVICE)
        segundos_carga = time.perf_counter() - t_carga
        print("Etiquetas originales:", model.config.id2label)
        parametros_m = model.num_parameters() / 1_000_000
        # Calentamiento, no incluido en el tiempo.
        for _ in range(3):
            clasificador("La atencion fue buena.", truncation=True, max_length=128)
        if GPU:
            torch.cuda.synchronize()
        predicciones = []
        latencias = []
        for texto in textos:
            if GPU:
                torch.cuda.synchronize()
            inicio = time.perf_counter()
            salida = clasificador(texto, truncation=True, max_length=128)
            if GPU:
                torch.cuda.synchronize()
            latencias.append((time.perf_counter() - inicio) * 1000)
            if isinstance(salida, list):
                salida = salida[0]
            predicciones.append(normalizar_etiqueta(salida["label"]))
        metricas = calcular_metricas(verdaderas, predicciones)
        pico_gpu = round(torch.cuda.max_memory_allocated() / 1024**3, 3) if GPU else None
        try:
            revisiones[modelo_id] = api.model_info(modelo_id).sha
        except Exception:
            revisiones[modelo_id] = "No disponible"
        filas_metricas.append({
            "modelo": spec["nombre"], "modelo_id": modelo_id,
            **metricas,
            "latencia_ms": round(float(np.mean(latencias)), 2),
            "parametros_millones": round(parametros_m, 1),
            "pico_gpu_gb": pico_gpu,
            "segundos_carga": round(segundos_carga, 1),
            "licencia": spec["licencia_revisada"],
            "revision": revisiones[modelo_id],
        })
        for idx, (real, pred) in enumerate(zip(verdaderas, predicciones)):
            filas_predicciones.append({"id": df.loc[idx, "id"], "modelo": spec["nombre"], "real": real, "predicho": pred})
        print("Accuracy:", round(metricas["accuracy"], 3), "F1 macro:", round(metricas["f1_macro"], 3), "Latencia ms:", round(float(np.mean(latencias)), 1))
    except Exception as exc:
        errores[modelo_id] = f"{type(exc).__name__}: {exc}"
        print("ERROR en este modelo:", errores[modelo_id])
    finally:
        for variable in ["clasificador", "model", "tokenizer"]:
            if variable in globals():
                globals().pop(variable, None)
        gc.collect()
        if GPU:
            torch.cuda.empty_cache()

resultados = pd.DataFrame(filas_metricas)
predicciones_df = pd.DataFrame(filas_predicciones)
resultados.to_csv("results/metricas_modelos.csv", index=False)
predicciones_df.to_csv("results/predicciones.csv", index=False)
if errores:
    Path("results/errores.txt").write_text(json.dumps(errores, ensure_ascii=False, indent=2), encoding="utf-8")
if len(resultados) != 3:
    print("ATENCION: solo se ejecutaron", len(resultados), "modelos. La comparacion NO esta completa. Revisar errores.txt.")
else:
    print("Comparacion completa: 3 modelos ejecutados.")
display(resultados.round(3))

## Paso 8. Comparar resultados y generar gráficos
No se usan números supuestos: los gráficos se crean exclusivamente a partir de las predicciones calculadas arriba.

In [ ]:
if resultados.empty:
    print("No hay resultados; revisar la celda de inferencia.")
else:
    ordenados = resultados.sort_values("f1_macro", ascending=False)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(ordenados["modelo"], ordenados["f1_macro"])
    ax.set_xlim(0, 1)
    ax.set_xlabel("F1 macro (0 a 1)")
    ax.set_title("Comparacion de F1 macro")
    fig.tight_layout()
    fig.savefig("results/grafico_f1.png", dpi=160)
    plt.show()

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(resultados["modelo"], resultados["latencia_ms"])
    ax.set_xlabel("Milisegundos por comentario (promedio)")
    ax.set_title("Latencia de inferencia")
    fig.tight_layout()
    fig.savefig("results/grafico_latencia.png", dpi=160)
    plt.show()

## Paso 9. Decisión técnica y registro de ejecución
Umbrales académicos propuestos: **F1 macro >= 0.75**, **recall NEG >= 0.80** y **latencia <= 300 ms** en GPU. Además, se exige licencia clara para una recomendación de uso empresarial. Si ninguno cumple, el prototipo no debe desplegarse. Un conjunto de 60 ejemplos ficticios no permite asegurar calidad en producción.

In [ ]:
if resultados.empty:
    recomendacion = "Sin recomendacion: no se obtuvieron resultados."
else:
    licencias_claras = {s["nombre"]: s["licencia_clara"] for s in MODELOS}
    candidatos = resultados[
        (resultados["f1_macro"] >= 0.75) &
        (resultados["recall_neg"] >= 0.80) &
        (resultados["latencia_ms"] <= 300) &
        (resultados["modelo"].map(licencias_claras).fillna(False))
    ]
    if candidatos.empty:
        recomendacion = "Ninguno cumple todos los umbrales y la condicion de licencia. No desplegar todavia."
    else:
        mejor = candidatos.sort_values(["f1_macro", "latencia_ms"], ascending=[False, True]).iloc[0]
        recomendacion = f"Modelo recomendado para el siguiente piloto: {mejor['modelo']} (F1 macro {mejor['f1_macro']:.3f}; latencia {mejor['latencia_ms']:.1f} ms)."
print(recomendacion)
Path("results/recomendacion.txt").write_text(recomendacion, encoding="utf-8")
metadatos = {
    "fecha_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "datasets": datasets.__version__,
    "huggingface_hub": huggingface_hub.__version__,
    "gpu_activada": GPU,
    "gpu": torch.cuda.get_device_name(0) if GPU else "CPU",
    "semilla": 42,
    "tamano_dataset": len(df),
    "sha256_dataset": DATOS_SHA256,
    "modelos_revision": revisiones,
    "errores": errores,
    "metodo_latencia": "60 solicitudes individuales, 3 calentamientos, sin descarga/carga",
    "longitud_maxima_tokens": 128,
}
Path("results/metadatos_ejecucion.json").write_text(json.dumps(metadatos, indent=2, ensure_ascii=False), encoding="utf-8")
print("Metadatos guardados en results/metadatos_ejecucion.json")

## Paso 10. Crear anexo Word con las métricas reales
Este anexo se genera **solo después de ejecutar los modelos**. Se puede anexar al reporte comparativo principal sin copiar los resultados a mano.

In [ ]:
from docx import Document
from docx.shared import Pt
anexo = Document()
anexo.add_heading("Anexo - Resultados medidos en Google Colab", 0)
anexo.add_paragraph("Proyecto: Supermercados Miroslava. Dataset ficticio de 60 comentarios en espanol. Las cifras siguientes se calcularon al ejecutar los modelos; no son estimaciones.")
anexo.add_paragraph("Equipo: " + (torch.cuda.get_device_name(0) if GPU else "CPU") + ". Fecha UTC: " + metadatos["fecha_utc"])
cols = ["Modelo", "Accuracy", "Precision macro", "Recall macro", "F1 macro", "Recall NEG", "Latencia (ms)"]
tabla = anexo.add_table(rows=1, cols=len(cols))
tabla.style = "Table Grid"
for c, nombre in zip(tabla.rows[0].cells, cols): c.text = nombre
for _, r in resultados.iterrows():
    vals = [r["modelo"], f"{r['accuracy']:.3f}", f"{r['precision_macro']:.3f}", f"{r['recall_macro']:.3f}", f"{r['f1_macro']:.3f}", f"{r['recall_neg']:.3f}", f"{r['latencia_ms']:.1f}"]
    for c, v in zip(tabla.add_row().cells, vals): c.text = str(v)
anexo.add_paragraph("Conclusion automatica: " + recomendacion)
anexo.add_paragraph("Limitacion: dataset simulado y pequeno; las metricas no demuestran resultados con clientes reales. Revisar las licencias antes de uso comercial.")
anexo.save("results/Anexo_metricas_ejecutadas.docx")
print("Anexo creado: results/Anexo_metricas_ejecutadas.docx")

## Paso 11. Descargar resultados
Al terminar, abre el enlace `resultados_colab.zip`. Incluye CSV, gráficos, metadatos y el anexo Word. Después puedes subirlos a la carpeta `results` de tu repositorio de GitHub.

In [ ]:
import shutil
shutil.make_archive("resultados_colab", "zip", "results")
from IPython.display import FileLink, display
display(FileLink("resultados_colab.zip"))
print("Proceso terminado. Revisa que metricas_modelos.csv tenga tres filas.")

### Conclusiones parciales
1. La comparación es justa porque los tres modelos reciben los mismos textos y etiquetas.
2. Un modelo más grande puede tardar más; la precisión debe medirse, no suponerse.
3. La recomendación final requiere comprobar **métricas, latencia, licencia y calidad con datos reales**. El dataset ficticio es solo una prueba de concepto.

**Fuentes de modelos:**
- https://huggingface.co/pysentimiento/robertuito-sentiment-analysis
- https://huggingface.co/lxyuan/distilbert-base-multilingual-cased-sentiments-student
- https://huggingface.co/cardiffnlp/twitter-xlm-roberta-base-sentiment